# Lección 02: ¿Cómo aprende la IA?

Entrenamos modelos, evaluamos con *train/test split*, visualizamos fronteras de decisión y exploramos el sobreajuste.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=2, suppress=True)

## Cargar Iris con 2 features

Usamos las dos últimas features (longitud y ancho del pétalo) para poder visualizar en 2D.

In [ ]:
iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target
target_names = iris.target_names
feature_names = [iris.feature_names[2], iris.feature_names[3]]

print(f'Muestras: {X.shape[0]}')
print(f'Features: {feature_names}')
print(f'Especies: {target_names}')

unique, counts = np.unique(y, return_counts=True)
for name, count in zip(target_names, counts):
    print(f'  {name}: {count} ({count/len(y)*100:.1f}%)')

## División en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')
print(f'Clases en train: {np.bincount(y_train)}')
print(f'Clases en test: {np.bincount(y_test)}')

## KNN con diferentes valores de k

In [ ]:
k_values = [1, 3, 5, 10, 15, 20, 30]
results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, knn.predict(X_train))
    test_acc = accuracy_score(y_test, knn.predict(X_test))
    results.append((k, train_acc, test_acc))

print(f"{'k':>3} | {'Train acc':>10} | {'Test acc':>10}")
print('-' * 32)
for k, train_acc, test_acc in results:
    print(f'{k:>3} | {train_acc:>10.3f} | {test_acc:>10.3f}')

## Visualización de fronteras de decisión

In [ ]:
def plot_decision_boundary(X, y, k, title, ax):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Set2)

    colors = ['#e74c3c', '#2ecc71', '#3498db']
    for cls, color, name in zip([0, 1, 2], colors, target_names):
        ax.scatter(X[y == cls, 0], X[y == cls, 1],
                   c=color, label=name, edgecolors='k', s=40, alpha=0.8)

    ax.set_xlabel(feature_names[0])
    ax.set_ylabel(feature_names[1])
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_decision_boundary(X_train, y_train, 1, 'KNN (k=1) - frontera irregular', axes[0])
plot_decision_boundary(X_train, y_train, 15, 'KNN (k=15) - frontera suave', axes[1])
plt.tight_layout()
plt.show()

## Regresión lineal sobre el target del Breast Cancer Wisconsin

In [ ]:
bcw = load_breast_cancer()
X_bcw = bcw.data[:, 0].reshape(-1, 1)
y_bcw = bcw.target

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bcw, y_bcw, test_size=0.3, random_state=42, stratify=y_bcw
)

lin = LinearRegression()
lin.fit(X_train_b, y_train_b)

print(f'MSE train (lineal): {mean_squared_error(y_train_b, lin.predict(X_train_b)):.4f}')
print(f'MSE test  (lineal): {mean_squared_error(y_test_b, lin.predict(X_test_b)):.4f}')

## Demostración de sobreajuste con polinomios

In [ ]:
degrees = range(1, 13)
train_errors, test_errors = [], []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train_b, y_train_b)

    train_errors.append(mean_squared_error(y_train_b, model.predict(X_train_b)))
    test_errors.append(mean_squared_error(y_test_b, model.predict(X_test_b)))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(degrees, train_errors, 'o-', label='Error de entrenamiento', color='#e74c3c')
ax.plot(degrees, test_errors, 's-', label='Error de prueba', color='#2ecc71')
ax.set_xlabel('Grado del polinomio')
ax.set_ylabel('Error cuadrático medio (MSE)')
ax.set_title('Subajuste vs. sobreajuste en BCW')
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.show()

## Resumen

- **Train/test split** es la forma más simple de estimar si un modelo generaliza a datos nuevos.
- **KNN** clasifica por cercanía; `k=1` memoriza el ruido y `k` grande suaviza la frontera.
- La **frontera de decisión** muestra cómo un algoritmo divide el espacio de features; no siempre es una línea recta.
- **Regresión lineal** encuentra una relación numérica; con polinomios de alto grado podemos reducir mucho el error de entrenamiento pero empeorar el de prueba.
- El **sobreajuste** aparece cuando el error de entrenamiento es bajo pero el de prueba es alto: el modelo memorizó en lugar de aprender.